`%reload_ext` loads the extension on first run and reloads it on subsequent runs, so re-running this cell always picks up any changes to the autoreload extension.

In [1]:
%reload_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import build123d as bd
from build123d import (
    Box,
    Compound,
    Cone,
    Cylinder,
    Part,
    Pos,
    ShapeList,
    export_stl,
    extrude,
)
from ocp_vscode import (
    get_defaults,
    reset_show,
    set_defaults,
    set_port,
    show,
    show_object,
)

import print_scripts_tree_d.shapes as shapes
from print_scripts_tree_d.export import save_stl

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("build123d").setLevel(logging.WARNING)
models = Path("models")
models.mkdir(exist_ok=True)
reset_show()
set_port(3939)

## Box with Cylinder Clip

In [2]:
from typing import cast

from build123d import Compound, Pos, Rot, ShapeList

from print_scripts_tree_d.params import CylinderClipParams, RoundedBoxParams

cp = CylinderClipParams()

bp = RoundedBoxParams(
    length=90.0,
    width=15.0,
    height=6.0,
    wall_thickness=4.0,
    corner_radius=2.5,
    top_fillet_radius=5.0,
    bottom_fillet_radius=1.0,
)
box = shapes.make_rounded_box(
    length=bp.length,
    width=bp.width,
    height=bp.height,
    wall_thickness=bp.wall_thickness,
    corner_radius=bp.corner_radius,
    top_fillet_radius=bp.top_fillet_radius,
    bottom_fillet_radius=bp.bottom_fillet_radius,
)
show(box)

Filleting top rim r=1.900...
Filleting bottom rim r=1.000...


+

In [3]:
clip = shapes.make_cylinder_clip(
    bore_diameter=cp.bore_diameter,
    body_depth=cp.body_depth,
    wall_thickness=cp.wall_thickness,
    flange_overlap=cp.flange_overlap,
    flange_thickness=cp.flange_thickness,
    tab_count=cp.tab_count,
    tab_protrusion=cp.tab_protrusion,
    tab_length=cp.tab_length,
    tab_width=cp.tab_width,
    slot_width=cp.slot_width,
    clearance=cp.clearance,
    flat_bottom=cp.flat_bottom,
    flat_fillet_r=1,
    flat_inner_margin=-0.6,
)
show(clip)

Building clip body OD=10.6 ID=8.2 depth=10.0...
Cutting 4 spring-finger slots...
Adding 4 snap tabs...


+


## Clip overhang check (FDM, arc fitting)

Clip in print orientation (`Rot(0, 90, 0)`, bed at Z=0). Curved surfaces self-support as **arcs**, so they're judged by geometry rather than a flat 45° cutoff: each circular feature is fit to its radius `R`, and from the layer height we find the near-horizontal cap that's too flat to self-support (per-layer step > `MAX_STEP`). The chord of that cap is the **top bridge span** — a hole prints clean if it's short. Flat (planar) downward faces have no arc and are flagged when their own per-layer step exceeds `MAX_STEP`. The bore renders **gold**, flat overhangs **red**.

In [4]:
# --- FDM overhang check (arc-fitting model) ---
# Curved surfaces self-support as arcs, so judge them by geometry, not a flat
# 45deg cutoff. For each circular feature we fit the arc (radius R) and, from
# the layer height, find the near-horizontal cap too flat to self-support
# (per-layer step > MAX_STEP); its chord is the top bridge span. A hole is
# fine if that bridge is short. Flat (planar) downward faces have no arc and
# are flagged when their own per-layer step exceeds MAX_STEP.
import math

from build123d import GeomType, Rot

LAYER_H = 0.2      # print layer height (mm)
MAX_STEP = 0.4     # max self-supporting horizontal step per layer (mm)
MAX_BRIDGE = 10.0  # longest clean unsupported bridge (mm)

cp_clip = Rot(0, 90, 0) * clip         # print orientation: bed Z0, build +Z
phi_c = math.atan(LAYER_H / MAX_STEP)  # half-angle of the unsupported cap
zmin = cp_clip.bounding_box().min.Z


def step_per_layer(nz):
    nz = max(-1.0, min(0.0, nz))
    return LAYER_H * (-nz) / math.sqrt(max(1e-12, 1 - nz * nz))


# Cylinders share the horizontal insertion axis through the origin, so a
# face normal pointing toward that axis (negative radial dot) is a bore.
holes, flat_overhang, seen = [], [], {}
for f in cp_clip.faces():
    p = f.position_at(0.5, 0.5)
    n = f.normal_at(p)
    if f.geom_type == GeomType.CYLINDER:
        kind = "hole" if (n.Y * p.Y + n.Z * p.Z) < 0 else "boss"
        seen[(round(f.radius, 1), kind)] = 2 * f.radius * math.sin(phi_c)
        if kind == "hole":
            holes.append(f)
    elif f.geom_type == GeomType.PLANE:
        if (
            n.Z < 0
            and step_per_layer(n.Z) > MAX_STEP
            and f.bounding_box().min.Z > zmin + 0.5
        ):
            flat_overhang.append(f)

print("Arc features:")
for (R, kind), span in sorted(seen.items()):
    if kind == "hole":
        verdict = "OK" if span <= MAX_BRIDGE else "NEEDS SUPPORT"
        print(f"  R={R:4.1f} hole -> top bridge {span:4.1f} mm  [{verdict}]")
    else:
        print(f"  R={R:4.1f} boss -> convex, self-supporting")
print(f"{len(flat_overhang)} flat downward face(s) need support")

reset_show()
show(
    cp_clip,
    ShapeList(holes),
    ShapeList(flat_overhang),
    names=["clip", "bore (arc)", "flat overhang"],
    colors=["lightgray", "gold", "red"],
    alphas=[0.55, 0.5, 1.0],
)

Arc features:
  R= 4.1 hole -> top bridge  3.7 mm  [OK]
  R= 5.3 boss -> convex, self-supporting
  R= 8.5 boss -> convex, self-supporting
5 flat downward face(s) need support
+++++++


In [5]:
tx = bp.length / 2 + cp.body_depth / 2
clip_rotated = Rot(0, 90, 0) * clip
# Derive z_offset from the actual clip geometry so it stays correct
# regardless of which flat_inner_margin was used to build the clip.
flat_bottom_z = clip_rotated.bounding_box().min.Z
z_offset = -flat_bottom_z - bp.height / 2
clip_at_end = Pos(tx, 0, z_offset) * clip_rotated

result = box + clip_at_end
if isinstance(result, ShapeList):
    assembly = Compound(children=list(result))
else:
    assembly = cast(Compound, result)

# Lift so both flat faces sit on Z = 0.
assembly = cast(Compound, Pos(0, 0, bp.height / 2) * assembly)

reset_show()
show(assembly)

+


In [6]:
save_stl(assembly, str(models / "box_with_clip.stl"))

Exported models\box_with_clip.stl
box_with_clip.stl validated — watertight, volume=4825.5 mm³


## Column

In [ ]:
col_body = Cylinder(1.5, 50)
col_foot = Cone(bottom_radius=1, top_radius=1.5, height=3)
column = shapes.make_column(
    body=col_body,
    height=50,
    foot=col_foot,
    diameter=3,
    gusset_size=3,
    gusset_thickness=1.5,
    gusset_position_z="top",
    gusset_orientation_xy=(0,120,240)
)
show(column)

## Table top

In [ ]:
table_top = shapes.make_hexagonal_mesh(
    length=6,
    width=6,
    thickness=1,
    hex_radius=4,
    spacing=1,
    outer_border=1,
)
show(table_top)

## Table assembly

In [ ]:
table = shapes.make_table(
    table_top=table_top,
    columns=[column.rotate(angle=180, axis=bd.Axis.X)],
    column_positions=[(50, 50)],
)
reset_show()
show(table)

In [ ]:
save_stl(table, str(models / "table.stl"))

## Rounded Box

In [ ]:
from print_scripts_tree_d.params import RoundedBoxParams

p = RoundedBoxParams()
box = shapes.make_rounded_box(
    length=p.length,
    width=p.width,
    height=p.height,
    wall_thickness=p.wall_thickness,
    corner_radius=p.corner_radius,
    top_fillet_radius=0.8,
    bottom_fillet_radius=0.5,
)
reset_show()
show(box)

In [ ]:
save_stl(box, str(models / "rounded_box.stl"))